# Extraction des données brutes



In [3]:
from pathlib import Path
import pymupdf as pypdf

import polars as pl

In [2]:
CORPUS = Path('data/veltis')

## Premier pdf

On arrive à bien extraire le texte mais on repère déjà un premier problème. 
1. Les entêtes reviennent à chaque page et on ne peut les distinguer du reste du document.
2. On ne reconnaît pas les titres
3. Les retours à la ligne suivent la largeur de la page.

In [4]:
doc = pypdf.open(CORPUS / 'presentation_veltis.pdf')
print(doc.page_count)

page = doc[0]
print(page.get_text())

2
Veltis SA — Présentation de l'entreprise
Document interne Veltis SA — diffusion restreinte
Page 1
Présentation de l'entreprise
Veltis SA — édition 2026
Qui sommes-nous ?
Fondée en 2011 à Valmont par Claire Dumont et Hugo Lefèvre, Veltis conçoit et fabrique des vélos cargo
à assistance électrique. Nos vélos transportent des enfants, des colis, des outils et parfois même des
cafés ambulants. Tout est dessiné, soudé et assemblé dans nos ateliers de Valmont.
Fin 2025, Veltis comptait 182 collaboratrices et collaborateurs, et ses vélos étaient distribués dans 11
pays.
Notre mission
Rendre le transport lourd léger : remplacer la voiture et la camionnette pour les trajets urbains de moins de
15 kilomètres, sans compromis sur la sécurité ni sur la charge transportée.
Nos valeurs
•
Robustesse : un vélo Veltis est conçu pour durer quinze ans et se répare.
•
Proximité : nous fabriquons localement et connaissons nos clients par leur nom.
•
Sobriété : chaque gramme et chaque watt doivent servir à

## Dans le détail

Les pdf sont en réalités découpes en plusieurs sous unités. D'abord des blocs, suivis de lignes et enfin des spans. Les spans regroupent du texte qui partagent la même police et la même taille.


In [11]:
spans = []
for bloc in page.get_text('dict').get('blocks'):
    for line in bloc.get('lines', []):
        for span in line.get('spans'):
            spans.append(span)

for span in spans[:10]:
    print(f'{span.get('font')} - {span.get('size')}: {span.get('text')}')


Helvetica - 8.0: Veltis SA — Présentation de l'entreprise
Helvetica - 8.0: Document interne Veltis SA — diffusion restreinte
Helvetica - 8.0: Page 1
Helvetica-Bold - 22.0: Présentation de l'entreprise
Helvetica - 12.0: Veltis SA — édition 2026
Helvetica-Bold - 16.0: Qui sommes-nous ?
Helvetica - 10.0: Fondée en 2011 à Valmont par Claire Dumont et Hugo Lefèvre, Veltis conçoit et fabrique des vélos cargo
Helvetica - 10.0: à assistance électrique. Nos vélos transportent des enfants, des colis, des outils et parfois même des
Helvetica - 10.0: cafés ambulants. Tout est dessiné, soudé et assemblé dans nos ateliers de Valmont.
Helvetica - 10.0: Fin 2025, Veltis comptait 182 collaboratrices et collaborateurs, et ses vélos étaient distribués dans 11


In [ ]:
# Pour chercher un titre
for span in spans:
    if span.get('size') > 10:
        print(span.get('text'))

## Oui mais

Travailler de cette manière est assez fragile.

Regardons sur un autre document.

In [16]:
newsletter = pypdf.open(CORPUS / 'newsletter_le_fil_2026-02.pdf')[0]

for bloc in newsletter.get_text('dict')['blocks']:
    for ligne in bloc.get('lines', []):
        for span in ligne['spans']:
            if span['size'] > 10:
                print(f'{span['size']:>4.0f} | {span['text']}')

  26 | Le Fil Veltis
  11 | Édito
  11 | Bienvenue aux nouveaux arrivants
  11 | Agenda
  11 | Portrait : Karim Benali, douze ans
  11 | d'atelier
  11 | Le saviez-vous ?
  22 | 1 000
  11 | Retour sur la journée portes
  11 | ouvertes


## Les tableaux

Il n'est pas rare dans un pdf d'avoir des images ou bien des tableaux.

Pymupdf a un outil pour les retrouver.

In [28]:
salaries = pypdf.open(CORPUS / 'grille_salariale_2026.pdf')[0]

tabs = salaries.find_tables()
print(f'{len(list(tabs))} found')

for tab in tabs:
    print(tab.extract())

2 found
[['Catégorie', 'Classe', 'Salaire mensuel brut de base', None, None, None], [None, None, 'Échelon 1\n(0-2 ans)', 'Échelon 2\n(3-5 ans)', 'Échelon 3\n(6-10 ans)', 'Échelon 4\n(plus de 10\nans)'], ['', 'A — Opérateur', '2 280 €', '2 390 €', '2 520 €', '2 660 €'], ['Ouvriers et ouvrières', 'B — Opérateur qualifié', '2 540 €', '2 670 €', '2 810 €', '2 960 €'], ['Employés et', 'C — Technicien, assistant', '3 150 €', '3 310 €', '3 480 €', '3 660 €'], ['employées', 'D — Technicien expert', '3 520 €', '3 700 €', '3 890 €', '4 090 €'], ['', "E — Responsable d'équipe", '4 200 €', '4 450 €', '4 720 €', '5 000 €'], ['Cadres', 'F — Responsable de service', '5 100 €', '5 420 €', '5 760 €', '6 120 €']]
[['Avantage', 'Bénéficiaires', 'Montant ou conditions'], ['Titres-repas', 'Tout le personnel', '8 € par jour travaillé'], ['Indemnité vélo', 'Trajets domicile-travail à vélo', '0,35 € par kilomètre, plafonnée à 40 km par jour'], ['Assurance hospitalisation', 'Tout le personnel et la famille', '

## Les images

La méthode ``get_text()`` ne renvoie pas les images mais seulement leur légende si disponnible. Et si un PDF est un scan par exemple, cette méthode pourrait ne rien renvoyer du tout.

In [31]:
org = pypdf.open(CORPUS / 'organigramme_et_processus_production.pdf')

images = []
for page in org:
    images.append(page.get_images(full=True))
images


[[(4,
   5,
   1520,
   598,
   8,
   'DeviceRGB',
   '',
   'FormXob.e48e4fc3dd8aac57bdf092a15039884d',
   'ASCII85Decode',
   0)],
 [(8,
   9,
   1520,
   376,
   8,
   'DeviceRGB',
   '',
   'FormXob.6d2f12e8ed54ac9ec0b60bdbe9c9bcf2',
   'ASCII85Decode',
   0)]]

In [42]:
image_folder = Path(CORPUS / 'images')
image_folder.mkdir(exist_ok=True)

for image in images:
    img_id = image[0][0]
    metadata = org.extract_image(img_id)
    print(metadata)
    img_name = f'{img_id}.{metadata.get('ext')}'
    print(img_name)
    (image_folder / img_name).write_bytes(metadata.get('image'))

{'width': 1520, 'height': 598, 'ext': 'png', 'colorspace': 3, 'xres': 96, 'yres': 96, 'bpc': 8, 'size': 49518, 'image': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x05\xf0\x00\x00\x02V\x08\x02\x00\x00\x00\x98\xba1]\x00\x00\x00\tpHYs\x00\x00\x0e\xc4\x00\x00\x0e\xc4\x01\x95+\x0e\x1b\x00\x00\xc1 IDATx\x9c\xec\xdd\x87_\x14\xd7\xfe\xff\xf1\xdf\x7f\x92\xdb\xef\xf7w\xef\xefkz\xb9)\xde\x94\x9b\xdc\x9b\xdcT\x93\x9b\xdc\x04{\x8dI,I4\xf6\x16\x8d%\xf6\x12kb\xefb/Q\xb1b\x03AADA\x01A@\x8a\x80tXX\x08\xbf\x8f\x9cx\x1cw\x97eYefg\xe7\xf5||\x1e>`8;\xfd\xcc\xee\xbc\x9d\x99\xfd?M\x00\x00\x00\x00\x00\x00\xb0\x95\xffc\xf5\x0c\x00\x00\x00\x00\x00\x00\xa0m\x08t\x00\x00\x00\x00\x00\x00l\x86@\x07\x00\x00\x00\x00\x00\xc0f\x08t\x00\x00\x00\x00\x00\x00l\x86@\x07\x00\x00\x00\x00\x00\xc0f\x08t\x00\x00\x00\x00\x00\x00l\x86@\x07\x00\x00\x00\x00\x00\xc0f\x08t\x00\x00\x00\x00\x00\x00l\x86@\x07\x00\x00\x00\x00\x00\xc0f\x08t\x00\x00\x00\x00\x00\x00l\x86@\x07\x00\x00\x00\x00\x00\xc0f\x08t\x00\x00\x00\x00\x00\x00l\x86@\x07

## Les faux amis

Attention que graphiques, des schémas qui seraient par exemple exportés depuis power point seront en fait des traits dans le PDF. Dans ces cas, le ni ``get_text()`` ni ``get_images`` ne vous aideront.

Il ne vous reste plus que le ``get_pixmap`` qui transforme toute une page en une seule image. Seul un OCR ou bien un modèle avec mmproj peut retrouver les infos contenues dedans.